# 06 Linear Baseline vs. Gradient-Boosted Trees


A logistic-regression baseline is evaluated on the exact same validation customers as the saved LightGBM model.


In [3]:
from pathlib import Path

def find_project_root() -> Path:
    """Use the parent of the notebook working directory as the project root."""
    project_root = Path.cwd().resolve().parent
    if not (project_root / 'data').is_dir():
        raise FileNotFoundError(
            "Expected the project data directory one level above the current working directory."
        )
    return project_root

import gc, json, platform, time
import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import roc_auc_score
print('Python:', platform.python_version())


Python: 3.10.19


### 1. Load the reference data and LightGBM result

The saved LightGBM predictions provide both the benchmark score and the canonical validation customer IDs. Reusing artifacts avoids retraining.


In [5]:
ROOT = find_project_root()
STEP_DIR = ROOT / '06_model_comparison'
DATA_PATH = ROOT / '04.5_temporal_refined' / 'train_aggregated_temporal_refined.feather'
LGB_METRICS_PATH = ROOT / '04.5_temporal_refined' / 'temporal_refined_metrics.json'
LGB_PREDICTIONS_PATH = ROOT / '04.5_temporal_refined' / 'temporal_refined_validation_predictions.parquet'
MODEL_PATH = STEP_DIR / 'logistic_regression_baseline.joblib'
COEFFICIENT_PATH = STEP_DIR / 'logistic_regression_coefficients.csv'
PREDICTION_PATH = STEP_DIR / 'logistic_regression_validation_predictions.parquet'
METRICS_PATH = STEP_DIR / 'model_baseline_comparison_metrics.json'

data = pd.read_feather(DATA_PATH)
assert data['customer_ID'].is_unique
assert data.shape[1] - 2 == 1022
lightgbm_result = json.loads(LGB_METRICS_PATH.read_text())['temporal_refined']
lightgbm_predictions = pd.read_parquet(LGB_PREDICTIONS_PATH)
print(f'Data: {data.shape[0]:,} customers × {data.shape[1]-2:,} model features')
print('Target rate:', data['target'].mean())
print('Saved 04.5 LightGBM:', lightgbm_result)


Data: 458,913 customers × 1,022 model features
Target rate: 0.2589336105100531
Saved 04.5 LightGBM: {'roc_auc': 0.9622113842161928, 'amex_metric': 0.7926643597558998, 'best_iteration': 1451, 'training_seconds': 645.1120789579581, 'feature_count': 1022}


## 2. Reproduce the validation split exactly

The original stratified split is recreated with the same random state, then checked against the saved prediction IDs. This explicit alignment test prevents two models evaluated on different samples from appearing directly comparable.


In [ ]:
customer_ids = data['customer_ID'].copy()
y = data['target'].astype('int8').copy()
indices = np.arange(len(data))
train_idx, valid_idx = train_test_split(indices, train_size=.70, test_size=.30, random_state=42, stratify=y)
y_train = y.iloc[train_idx].to_numpy(); y_valid = y.iloc[valid_idx].to_numpy()
valid_ids = customer_ids.iloc[valid_idx].to_numpy()
assert np.array_equal(valid_ids, lightgbm_predictions['customer_ID'].to_numpy())
assert np.array_equal(y_valid, lightgbm_predictions['target'].to_numpy())
print(f'Train: {len(train_idx):,}; validation: {len(valid_idx):,}; LightGBM prediction alignment passed')


## 3. Fit preprocessing on training data only

Dates are represented as relative days. Numeric features are standardized using training-set means and variances; missing standardized values are filled with zero, equivalent to mean imputation in the original scale. Categorical last values are one-hot encoded with a vocabulary learned only from training customers.

Validation data never influences scaling or category mappings, preventing preprocessing leakage.


In [ ]:
data['S_2_last'] = (pd.to_datetime(data['S_2_last']) - pd.Timestamp('2018-03-01')).dt.days.astype('float32')
CATEGORICAL_VARIABLES = ['B_30','B_38','D_114','D_116','D_117','D_120','D_126','D_63','D_64','D_66','D_68']
categorical_columns = [f'{c}_last' for c in CATEGORICAL_VARIABLES]
feature_columns = [c for c in data.columns if c not in ['customer_ID','target']]
numeric_columns = [c for c in feature_columns if c not in categorical_columns]
BATCH_SIZE = 20_000

def categorical_frame(row_indices):
    return data.iloc[row_indices][categorical_columns].astype('string').fillna('__MISSING__')

encoder = OneHotEncoder(handle_unknown='ignore', dtype=np.float32, sparse_output=True)
encoder.fit(categorical_frame(train_idx))
scaler = StandardScaler()
for start in range(0, len(train_idx), BATCH_SIZE):
    rows = train_idx[start:start+BATCH_SIZE]
    scaler.partial_fit(data.iloc[rows][numeric_columns].to_numpy(dtype='float32', copy=True))
print('Numeric features:', len(numeric_columns))
print('One-hot features:', len(encoder.get_feature_names_out(categorical_columns)))


## 4. Train logistic regression in mini-batches

`SGDClassifier(loss='log_loss')` optimizes the logistic objective over five shuffled epochs. Each batch is transformed just before use, keeping memory bounded.

The fixed random seed makes batch order reproducible. This model is intended as a strong, scalable linear baseline rather than an exhaustive regularization search.


In [ ]:
def transform_rows(row_indices):
    numeric = data.iloc[row_indices][numeric_columns].to_numpy(dtype='float32', copy=True)
    numeric = scaler.transform(numeric).astype('float32', copy=False)
    np.nan_to_num(numeric, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
    numeric_sparse = sparse.csr_matrix(numeric)
    categorical_sparse = encoder.transform(categorical_frame(row_indices))
    return sparse.hstack([numeric_sparse, categorical_sparse], format='csr', dtype=np.float32)

logistic = SGDClassifier(loss='log_loss', penalty='l2', alpha=1e-4, learning_rate='optimal', max_iter=1, tol=None, average=True, random_state=42)
EPOCHS = 5
rng = np.random.default_rng(42)
started = time.perf_counter()
for epoch in range(EPOCHS):
    epoch_rows = train_idx.copy(); rng.shuffle(epoch_rows)
    for start in range(0, len(epoch_rows), BATCH_SIZE):
        rows = epoch_rows[start:start+BATCH_SIZE]
        batch = transform_rows(rows)
        logistic.partial_fit(batch, y.iloc[rows].to_numpy(), classes=np.array([0,1], dtype='int8'))
        del batch
    print(f'Epoch {epoch+1}/{EPOCHS} complete')
training_seconds = time.perf_counter() - started
print(f'Training time: {training_seconds:.1f} sec')


## 5. Compare ranking performance

Both models are scored with ROC-AUC and the official AMEX metric on identical customers. LightGBM's clear advantage indicates that nonlinear thresholds, interactions, and missing-value patterns carry information that a single linear log-odds function does not capture.


In [ ]:
def amex_metric(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.int8); y_pred = np.asarray(y_pred, dtype=np.float64)
    weights = np.where(y_true == 0, 20.0, 1.0)
    order = np.argsort(-y_pred, kind='mergesort'); yy, ww = y_true[order], weights[order]
    top = np.cumsum(ww) <= .04 * ww.sum()
    top_four = (yy[top] * ww[top]).sum() / (y_true * weights).sum()
    def gini(pred):
        idx=np.argsort(-pred,kind='mergesort'); sy,sw=y_true[idx],weights[idx]
        return np.sum((np.cumsum(sy*sw)/np.sum(sy*sw)-np.cumsum(sw)/np.sum(sw))*sw)
    return .5 * (gini(y_pred)/gini(y_true)+top_four)

probability = np.empty(len(valid_idx), dtype='float32')
for start in range(0, len(valid_idx), BATCH_SIZE):
    rows = valid_idx[start:start+BATCH_SIZE]
    batch = transform_rows(rows)
    probability[start:start+len(rows)] = logistic.predict_proba(batch)[:,1]
    del batch
logistic_result = {'roc_auc':float(roc_auc_score(y_valid,probability)), 'amex_metric':float(amex_metric(y_valid,probability)), 'training_seconds':float(training_seconds), 'feature_count_after_encoding':int(logistic.coef_.shape[1]), 'epochs':EPOCHS}
comparison = pd.DataFrame([logistic_result,lightgbm_result], index=['Logistic Regression','LightGBM 04.5'])
display(comparison); logistic_result


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Same-customer comparison of linear and nonlinear models
model_scores = pd.DataFrame([
    {'model': 'Logistic regression', 'ROC-AUC': logistic_result['roc_auc'], 'AMEX metric': logistic_result['amex_metric']},
    {'model': 'LightGBM 04.5', 'ROC-AUC': lightgbm_result['roc_auc'], 'AMEX metric': lightgbm_result['amex_metric']},
]).melt(id_vars='model', var_name='metric', value_name='score')
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(data=model_scores, x='metric', y='score', hue='model', palette=['#9CA3AF', '#4C78A8'], ax=ax)
ax.set(title='LightGBM captures signal beyond a linear decision boundary', xlabel='', ylabel='Validation score', ylim=(0.72, 0.98))
ax.legend(title='')
sns.despine()
plt.tight_layout()
plt.show()


## 6. Inspect coefficients and save artifacts

Standardized numeric coefficients and one-hot category coefficients provide a transparent view of the strongest linear associations. Their signs describe the fitted model, not causal effects.

The fitted estimator, validation predictions, coefficient table, and comparison metrics are saved for reproducibility.


In [ ]:
encoded_names = encoder.get_feature_names_out(categorical_columns).tolist()
model_feature_names = numeric_columns + encoded_names
assert len(model_feature_names) == logistic.coef_.shape[1]
coefficients = pd.DataFrame({'feature':model_feature_names,'coefficient':logistic.coef_[0]})
coefficients['abs_coefficient'] = coefficients['coefficient'].abs()
coefficients = coefficients.sort_values(['abs_coefficient','feature'],ascending=[False,True]).reset_index(drop=True)
coefficients['rank'] = np.arange(1,len(coefficients)+1)
coefficients.to_csv(COEFFICIENT_PATH,index=False)
display(coefficients.head(30))

joblib.dump({'model':logistic,'scaler':scaler,'encoder':encoder,'numeric_columns':numeric_columns,'categorical_columns':categorical_columns}, MODEL_PATH, compress=3)
pd.DataFrame({'customer_ID':valid_ids,'target':y_valid,'logistic_probability':probability,'lightgbm_probability':lightgbm_predictions['temporal_refined_probability'].to_numpy()}).to_parquet(PREDICTION_PATH,index=False,compression='zstd')
metrics={'logistic_regression':logistic_result,'lightgbm_04_5':lightgbm_result,'same_validation_customers':True,'preprocessing':'train-only StandardScaler + mean imputation + one-hot encoding','optimizer':'SGD, log-loss, 5 epochs'}
METRICS_PATH.write_text(json.dumps(metrics,indent=2),encoding='utf-8')
for path in [MODEL_PATH,COEFFICIENT_PATH,PREDICTION_PATH,METRICS_PATH]: print(path.name,'->',path.resolve())
del data; gc.collect()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Strongest standardized linear associations
coefficient_plot = pd.concat([
    coefficients.nsmallest(10, 'coefficient'),
    coefficients.nlargest(10, 'coefficient'),
]).sort_values('coefficient')
fig, ax = plt.subplots(figsize=(9, 7))
colors = ['#4C78A8' if value < 0 else '#E45756' for value in coefficient_plot['coefficient']]
ax.barh(coefficient_plot['feature'], coefficient_plot['coefficient'], color=colors)
ax.axvline(0, color='0.35', linewidth=1)
ax.set(title='Largest logistic-regression coefficients', xlabel='Coefficient on standardized / one-hot feature', ylabel='')
sns.despine()
plt.tight_layout()
plt.show()
